# District_Heatmap_Example

Notebook companion to `generate_district_heatmap.py` -- "the forest, not the trees" (#3 in this series, after `CalStateTesting_Example` and `SES_Achievement_Gap_Terrain_Example`): every San Diego Unified school as one flat, color-animated marker laid directly over the district's real satellite image, color-coded by CST proficiency and animated across all 10 years (2003-2012) via GlyphViz's Channels system.

This is the first real-data generator in this repo to actually drive Channels (`ch-map`/`ch-tracks`) end-to-end -- still a pending high-priority feature on the project roadmap. Color is a traffic-light gradient (red=low proficiency, yellow=mid, green=high); confirmed via headless render at frames 0/90/180/270 that the whole map shifts smoothly and continuously from orange-dominant (2003) to green-dominant (2012), with a few persistent red/orange outliers throughout -- a real, data-backed "the district improved, but unevenly" story.

**A real data-quality bug was caught and fixed here**: subgroup=1 ("All Students")'s own `percentaboveproficient` is a literal `0.0` placeholder in the source database for every school, 2003-2009 -- not a real score. The default `--subgroup-codes` is therefore `3 4` (Males+Females summed together, confirmed to match subgroup=1's population almost exactly) rather than `1` itself, and both fetchers treat any (year, school) cell with a real student count but an exactly-zero weighted score as "not actually computed" so the animation holds flat at the nearest real year instead of showing a false low score. See the module docstring's "Known data gap" section for the full investigation.

`--subgroup-codes` generalizes this to any other subgroup's own geographic trend (e.g. just `31` for Economically Disadvantaged).

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from generate_district_heatmap import build_arg_parser, generate

## Parameters

Edit this list and re-run -- same flags as the CLI (`python generate_district_heatmap.py --help`). `--source mysql` requires a reachable MySQL server with `calstatetesting_all`; `--source csv` reads `sdusd<year>.csv` files from `--data-dir` instead.

In [ ]:
cli_args = [
    "--source", "mysql",
    "--start-year", "2003",
    "--end-year", "2012",
    "--prefix", "District_Heatmap_Example_notebook",
]

args = build_arg_parser().parse_args(cli_args)
args

## Generate the node/tag/channel CSVs

In [ ]:
node_path, tag_path = generate(args)
node_path, tag_path

## Peek at the output: biggest movers over the decade

In [ ]:
import pandas as pd

tags_df = pd.read_csv(tag_path)

# Tag titles look like "Lincoln High: 22.1% (2003) -> 54.6% (2012)"
parsed = tags_df["title"].str.extract(
    r"(?P<school>.+): (?P<pct_start>[\d.]+)% \((?P<year_start>\d+)\) -> (?P<pct_end>[\d.]+)% \((?P<year_end>\d+)\)"
)
parsed["pct_start"] = parsed["pct_start"].astype(float)
parsed["pct_end"] = parsed["pct_end"].astype(float)
parsed["change"] = parsed["pct_end"] - parsed["pct_start"]

print("Biggest gainers:")
display(parsed.sort_values("change", ascending=False).head(10))
print("\nBiggest decliners:")
display(parsed.sort_values("change").head(10))

## Optional: render a quick preview at a specific frame

Unlike the static examples, this one is animated -- `glyphviz_core/channel_engine.py`'s `ChannelEngine.apply_frame(frame)` mutates node colors in place for a given frame (0 = first year, last = `frame_count - 1`). `frames_per_year` (CLI default 30) tells you which frame index corresponds to which year.

In [ ]:
REPO_ROOT = HERE.parent.parent  # examples/District_Heatmap_Example -> repo root
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from PySide6.QtWidgets import QApplication

app = QApplication.instance() or QApplication(sys.argv[:1])

from glyphviz_core.csv_reader import load_node_csv
from glyphviz_core.channel_loader import load_ch_map, load_ch_tracks
from glyphviz_core.channel_engine import ChannelEngine
from glyphviz_gl.viewport import Viewport

nodes = load_node_csv(str(node_path))
# Construct paths directly rather than via find_channel_files()'s directory
# glob -- that helper requires exactly one ch-map/ch-track match in the
# folder, which breaks once more than one example's output csvs coexist here.
map_path = Path(str(node_path).replace("_gv_node.csv", "_gv_ch-map.csv"))
tracks_path = Path(str(node_path).replace("_gv_node.csv", "_gv_ch-tracks.csv"))
engine = ChannelEngine()
engine.load(load_ch_map(map_path), *load_ch_tracks(tracks_path), nodes)
print(f"frame_count={engine.frame_count}")

FRAME = engine.frame_count - 1   # last year; try 0 for the first year
engine.apply_frame(FRAME)

vp = Viewport()
vp.set_nodes(nodes)
vp.show_tags = False
vp.set_camera(azimuth=0, elevation=89, distance=300)   # top-down

preview_path = HERE / "notebook_preview.png"
vp.export_png(str(preview_path), width=960, height=540)

from IPython.display import Image
Image(filename=str(preview_path))

## Load in GlyphViz

File > Open Node CSV -> the `node_path` printed above, then press Play (Channels) to watch the decade unfold.